# D2 · Calibration: who tells the truth when they say 30%?

**Type**: Interpret · **Level**: ●●● challenge · **Time**: about 50 minutes · **extended path**

## Objective

Compare five models on the same data for **discrimination** (AUC) and **calibration** (expected calibration error ECE, Brier score, log-loss), read the calibration curves
and verify that a post-hoc calibration (`CalibratedClassifierCV`) fixes a random forest.

## Dataset

`ds.loans(n=6000, with_missing=False)`, stratified 50/50 split: `X_tr, X_te, y_te`. The predicted probabilities on the test set of the five models are in `probabilities`
(dictionary name → array): logistic, logistic with `class_weight="balanced"`, random forest, random forest calibrated with sigmoid and with isotonic.

## Educational purpose

Tell apart two different questions: "does it rank well?" (AUC) and "are the probabilities true?" (calibration). A model can have the same AUC as another and unusable probabilities.

## Task

1. Write `ece(y, p, k=10)`: **expected calibration error** with `k` groups of equal size (`pd.qcut(p, k, duplicates="drop")`): sum over the groups of `weight · |mean(p) − observed frequency|`, with `weight = share of examples in the group`.
2. Build `table` (index: the five models; columns `auc`, `brier`, `logloss`, `ece`, `mean_p`) with `roc_auc_score`, `brier_score_loss`, `log_loss`, your `ece` and the mean of the probabilities.
3. Answer with the data: `same_discrimination` (True if the logistic and the `balanced` logistic have AUCs differing by less than 0.005); `balanced_overestimation` (difference between `mean_p` of `balanced` and the true frequency of positives); `best_calibration` (name of the model with minimum ECE); `best_auc` (name of the model with maximum AUC).
4. Compute `isotonic_gain = ece(forest) / ece(isotonic_forest)` and `sigmoid_gain` (the same with the sigmoid): by how many times does the calibration error of the forest drop?
5. Question: `why_balanced_is_skewed = 'c'` (choose `'a'`, `'b'` or `'c'`): a) the model ranks badly; b) the model is overfitting; c) the weights shift the probabilities upward without changing (almost) the ordering.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = ds.loans(n=6000, with_missing=False)
df["ratio"] = df["amount"] / df["income"]
cols = ["ratio", "age", "job_tenure", "income", "amount"]
X_tr, X_te, y_tr, y_te = train_test_split(df[cols], df["defaulted"], test_size=0.5, stratify=df["defaulted"], random_state=0)
y_te = y_te.to_numpy()

models = {
    "logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "balanced": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")),
    "forest": RandomForestClassifier(300, random_state=0, n_jobs=-1),
    "forest_sigmoid": CalibratedClassifierCV(RandomForestClassifier(300, random_state=0, n_jobs=-1), method="sigmoid", cv=5),
    "forest_isotonic": CalibratedClassifierCV(RandomForestClassifier(300, random_state=0, n_jobs=-1), method="isotonic", cv=5),
}
probabilities = {name: m.fit(X_tr, y_tr).predict_proba(X_te)[:, 1] for name, m in models.items()}
print("frequency of positives in the test set:", y_te.mean().round(3))

## Your work

In [ ]:
def ece(y, p, k=10):
    ...

In [ ]:
table = pd.DataFrame(index=list(probabilities), columns=["auc", "brier", "logloss", "ece", "mean_p"], dtype=float)
table

In [ ]:
same_discrimination = None
balanced_overestimation = None
best_calibration = None
best_auc = None
isotonic_gain = None
sigmoid_gain = None
why_balanced_is_skewed = None

### The plot (to look at, not to be graded)

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
for name, p in probabilities.items():
    freq, mean = calibration_curve(y_te, p, n_bins=8, strategy="quantile")
    ax.plot(mean, freq, "o-", label=name, ms=4)
ax.plot([0, 1], [0, 1], "k:"); ax.set(xlabel="predicted probability", ylabel="observed frequency"); ax.legend(); plt.show()

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`groups = pd.qcut(p, k, duplicates="drop"); df_ = pd.DataFrame({"y": y, "p": p, "g": groups}); s = df_.groupby("g").agg(n=("y", "size"), y=("y", "mean"), p=("p", "mean"))`; `ece = (s["n"] / s["n"].sum() * (s["p"] - s["y"]).abs()).sum()`.'):
    def reference(y, p, k=10):
        d = pd.DataFrame({"y": y, "p": p, "g": pd.qcut(p, k, duplicates="drop")}).groupby("g", observed=True).agg(n=("y", "size"), y=("y", "mean"), p=("p", "mean"))
        return float((d["n"] / d["n"].sum() * (d["p"] - d["y"]).abs()).sum())
    check(close(ece(y_te, probabilities["balanced"]), reference(y_te, probabilities["balanced"])) and close(ece(y_te, probabilities["forest"]), reference(y_te, probabilities["forest"])), "correct ECE", "groups of equal size with pd.qcut, then a weighted sum of the absolute gaps")
    check(table.notna().all().all() and close(table.loc["logistic", "auc"], roc_auc_score(y_te, probabilities["logistic"])), "complete table", "five rows and five columns")
    check(same_discrimination is True and balanced_overestimation > 0.2, "AUC practically equal, but `balanced` overestimates the frequency by over 0.2", "compare AUC and mean_p of the two models")
    check(best_calibration == "logistic" and best_auc in ("logistic", "balanced"), "the plain logistic is the best calibrated, and the two logistics beat the forests on AUC", "idxmin of the ECE, idxmax of the AUC")
    check(isotonic_gain > 2 and sigmoid_gain > 1.5, "calibration reduces the ECE of the forest by over 2 times (isotonic) and 1.5 times (sigmoid)", "ece(forest) / ece(calibrated version)")
    check(table.loc["forest_isotonic", "logloss"] < table.loc["forest", "logloss"] and table.loc["forest_isotonic", "brier"] < table.loc["forest", "brier"], "calibrating improves log-loss and Brier too", "look at the logloss and brier columns")
    check(why_balanced_is_skewed == "c", "correct answer: the weights shift the probabilities", "the AUC is almost equal, so the ordering is the same")
done()

---
The commented solution is in `solutions/D_interpret/D2_calibration_of_five_models.ipynb`. Open it after you pass the check, or after you have really tried.